# 03 — Learning: `tok_split`, `token_accept`, `monad_learn_ex`, the A-matrix
C source: `tokenizer.c`; `filter.c`; `monad.c` lines 252–550 (`monad_a_add`, `dirac_pole`, `monad_learn_ex`). Port: `tools/monad_py.py`.
Checked against the compiled binary: same text → both write a state file → the files are compared (`tools/verify_learn_against_c.py`).

In [1]:
import os, sys, json, math, collections, re, hashlib
import numpy as np
sys.path.insert(0, os.path.abspath("../tools"))
import ptol_bin, monad_py as mp
ROOT  = os.path.abspath("..")
STORE = os.path.join(ROOT, "dirty", "monad_wordnet_v4_20260519.bin")
CBIN  = os.path.join(ROOT, "build", "PtolC_eb28ce8", "ptolemy")
SPACES = "[\u200b\u2009\u2008\u2007\u2006\u2005\u2004\u2003]"   # the eight Fermat separators

## Tokeniser
`isalpha(c) || c == '\''` on an `unsigned char`, C locale: **ASCII letters and apostrophe only**; everything else (digits, punctuation, every byte ≥ 128) ends a word.
Lower-cased; a token must be ≥ 2 characters and must not start with an apostrophe.

In [2]:
for s in ["The tree is a plant.", "arbre 木 baum δένδρο", "don't 'quoted' x ab", "ağaç dört"]:
    print(f"{s!r:28} -> {mp.tok_split(s)}")

'The tree is a plant.'       -> ['the', 'tree', 'is', 'plant']
'arbre 木 baum δένδρο'        -> ['arbre', 'baum']
"don't 'quoted' x ab"        -> ["don't", 'ab']
'ağaç dört'                  -> ['rt']


## `token_accept` (prose rules)

In [3]:
tests = ["tree","a","qgt","bcdfg","xxxxxxxxxxxxxxxxxxxxxxxxx","deadbeef","ban'","aeiou","schzr","strengths","qwrtypsdfghjkl"]
for t in tests: print(f"{t:28} accepted={mp.token_accept(t)}")

tree                         accepted=True
a                            accepted=False
qgt                          accepted=False
bcdfg                        accepted=False
xxxxxxxxxxxxxxxxxxxxxxxxx    accepted=False
deadbeef                     accepted=False
ban'                         accepted=False
aeiou                        accepted=True
schzr                        accepted=False
strengths                    accepted=False
qwrtypsdfghjkl               accepted=False


Among the 24 485 words the store displays, how many would today's prose filter reject? (They were admitted by the earlier, pre-vowel-fix filter or by the WordNet path.)

In [4]:
m = mp.Monad.from_bin(STORE)
rej = [v["word"] for v in m.vocab.values() if not mp.token_accept(v["word"])]
print(len(rej), "of", len(m.vocab), rej[:20])

1133 of 24485 ['tcpd', 'vqr', 'ghyc', 'zbk', 'db', 'hgc', 'fqb', 'plb', 'btlq', 'sdgm', 'vwc', 'rzs', 'dcs', 'cbw', 'xyyc', 'nkd', 'ycc', 'hwnd', 'qrdnb', 'kfhh']


## Learning one sentence, shown
For each accepted token: address → `β += E²·α·|sin γ|·π/2` (cap 7.552) → the slot displays the highest-`E` arrival. Then every pair within 12 accepted tokens gets
`w = Eᵢ·Eⱼ·couple / ((|γᵢ−γⱼ| + gap)·d)` where `d` is the text distance, `couple = 2` if both zeros share the Dirac pole sign (else 1), `gap = 0.000707`.

In [5]:
g = mp.Monad()
text = "The tree is a plant. Water is a liquid that the tree drinks."
g.learn(text)
print("accepted tokens -> slot, E, beta")
for w in mp.tok_split(text):
    if mp.token_accept(w):
        i, E = mp.word_coords(w); print(f"  {w:8} z#{i:<6} E={E:.4f} gamma={g.zeros[i]:10.3f} beta={g.beta[i]:.6f}  slot displays {g.vocab[i]['word']!r}")
A = g._A_dict(); print(len(A), "edges; strongest:")
for (i,j),w in sorted(A.items(), key=lambda kv:-kv[1])[:6]: print(f"  {g.vocab[i]['word']}({i}) <-> {g.vocab[j]['word']}({j})  w={w:.4e}   key=(i<<15)|j={(i<<15)|j}")

accepted tokens -> slot, E, beta
  the      z#841    E=0.2568 gamma=  1234.616 beta=0.000138  slot displays 'the'
  tree     z#3109   E=0.2859 gamma=  3642.834 beta=0.002612  slot displays 'tree'
  is       z#1894   E=0.2703 gamma=  2405.433 beta=0.002040  slot displays 'is'
  plant    z#21726  E=0.5251 gamma= 19403.402 beta=0.003536  slot displays 'plant'
  water    z#9361   E=0.3663 gamma=  9331.320 beta=0.001571  slot displays 'water'
  is       z#1894   E=0.2703 gamma=  2405.433 beta=0.002040  slot displays 'is'
  liquid   z#4354   E=0.3019 gamma=  4844.106 beta=0.000401  slot displays 'liquid'
  that     z#24362  E=0.5589 gamma= 21451.606 beta=0.003634  slot displays 'that'
  the      z#841    E=0.2568 gamma=  1234.616 beta=0.000138  slot displays 'the'
  tree     z#3109   E=0.2859 gamma=  3642.834 beta=0.002612  slot displays 'tree'
  drinks   z#14968  E=0.4383 gamma= 14015.877 beta=0.002922  slot displays 'drinks'
24 edges; strongest:
  is(1894) <-> tree(3109)  w=1.5617e-04   ke

`A` is stored once per unordered pair with `i < j`; the key packs both indices into 30 bits, which is only possible while `N < 2¹⁵ = 32 768` (N = 25 000 here).

## Verification against the compiled C binary (three texts, ground state each time)

In [6]:
import verify_learn_against_c as v
for t in ["The tree is a plant. Water is a liquid that the tree drinks. Arbre means tree in French.",
          "I am Ptolemy. I am named after Ptolemy the Second, Philadelphos, who commissioned the Septuagint.",
          "Holcus is a genus of grasses; velvet grass, Yorkshire fog. Qgt xz bcdfg aeiou 12345 http://x.y/z tree tree tree."]:
    print(v.compare(t))

{'vocab_equal': True, 'n_vocab': (12, 12), 'edges': (34, 34), 'edge_keys_equal': True, 'max_rel_edge_err': 0.0, 'max_abs_beta_err': 0.0, 'word_count': (16, 16)}
{'vocab_equal': True, 'n_vocab': (8, 8), 'edges': (28, 28), 'edge_keys_equal': True, 'max_rel_edge_err': 0.0, 'max_abs_beta_err': 0.0, 'word_count': (13, 13)}


{'vocab_equal': True, 'n_vocab': (11, 11), 'edges': (36, 36), 'edge_keys_equal': True, 'max_rel_edge_err': 0.0, 'max_abs_beta_err': 0.0, 'word_count': (13, 13)}


`vocab_equal`, `edge_keys_equal` true and both error columns 0.0: the port is exact on this path.